# Whisper Accent Robustness — Baseline Evaluation

Visualisation notebook. Run `eval_model_perf.py` on SLURM first to generate CSVs.
Covers both **scripted** (test split) and **spontaneous** speech.

In [1]:
# ── Config ────────────────────────────────────────────────────────────────────
RESULTS_DIR = "results/model_perf_comparison"

# Keys must match filenames: {model_key}_{split}_predictions.csv
MODEL_KEYS = {
    "baseline":      "Zero-shot",
    "baseline_lora": "Naive LoRA FT",
    "ctc_aux":       "CTC Aux",
}
SPLITS = ["scripted", "spontaneous"]


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, re
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from jiwer import wer as jiwer_wer, cer as jiwer_cer
from pathlib import Path

pd.set_option("display.max_colwidth", 80)


In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def compute_metrics_df(df, ref_col="reference_norm", pred_col="prediction_norm"):
    refs, preds = df[ref_col].fillna("").tolist(), df[pred_col].fillna("").tolist()
    return {"wer": jiwer_wer(refs, preds), "cer": jiwer_cer(refs, preds)}

def compute_grouped_metrics(df, group_col="speaker_native_language",
                             ref_col="reference_norm", pred_col="prediction_norm"):
    rows = []
    for grp, sub in df.groupby(group_col):
        refs, preds = sub[ref_col].fillna("").tolist(), sub[pred_col].fillna("").tolist()
        rows.append({group_col: grp, "num_utts": len(sub),
                     "wer": jiwer_wer(refs, preds), "cer": jiwer_cer(refs, preds)})
    return pd.DataFrame(rows)

def load_results(model_key, split):
    p = Path(RESULTS_DIR) / f"{model_key}_{split}_predictions.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p}  — run eval_model_perf.py first")
    return pd.read_csv(p)

print("Helpers loaded.")


Helpers loaded.


In [4]:
# ── Load all cached CSVs ──────────────────────────────────────────────────────
results = {}
for key in MODEL_KEYS:
    results[key] = {}
    for split in SPLITS:
        try:
            results[key][split] = load_results(key, split)
            print(f"  Loaded {key}/{split}: {len(results[key][split])} rows")
        except FileNotFoundError as e:
            print(f"  WARNING: {e}")


  Loaded baseline/scripted: 900 rows
  Loaded baseline/spontaneous: 22 rows
  Loaded baseline_lora/scripted: 900 rows
  Loaded baseline_lora/spontaneous: 22 rows
  Loaded ctc_aux/scripted: 900 rows
  Loaded ctc_aux/spontaneous: 22 rows


# Part 1 — Overall WER / CER

In [5]:
# ── Overall metrics table — all models × splits ───────────────────────────────
rows = []
for key, label in MODEL_KEYS.items():
    for split in SPLITS:
        if split not in results.get(key, {}): continue
        m = compute_metrics_df(results[key][split])
        rows.append({"Model": label, "Split": split,
                     "WER": m["wer"], "CER": m["cer"]})

overall_df = pd.DataFrame(rows)
display(overall_df.pivot(index="Model", columns="Split", values="WER")
        .style.format("{:.4f}").set_caption("WER by Model × Split"))
display(overall_df.pivot(index="Model", columns="Split", values="CER")
        .style.format("{:.4f}").set_caption("CER by Model × Split"))


Split,scripted,spontaneous
Model,,
CTC Aux,0.1173,0.6278
Naive LoRA FT,0.0768,0.6375
Zero-shot,0.1326,0.6349


Split,scripted,spontaneous
Model,,
CTC Aux,0.0610,0.6073
Naive LoRA FT,0.0406,0.6175
Zero-shot,0.0688,0.6128


In [11]:
# ── Overall WER bar charts — one per split ───────────────────────────────────
for split in SPLITS:
    sub = overall_df[overall_df["Split"] == split]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        name=split.capitalize(),
        x=sub["Model"].tolist(),
        y=sub["WER"].tolist(),
        text=[f"{v:.3f}" for v in sub["WER"]],
        textposition="outside",
    ))
    fig.update_layout(
        title=f"WER by Model — {split.capitalize()}",
        showlegend=False,
    )
    fig.update_yaxes(title_text="WER", tickformat=".0%")
    fig.show()


# Part 2 — Per-L1 WER

In [12]:
# ── Per-L1 WER — one subplot per split ───────────────────────────────────────
for split in SPLITS:
    rows_l1 = []
    for key, label in MODEL_KEYS.items():
        if split not in results.get(key, {}): continue
        grp = compute_grouped_metrics(results[key][split])
        grp["Model"] = label
        rows_l1.append(grp)
    if not rows_l1: continue

    l1_df = pd.concat(rows_l1, ignore_index=True)

    # Baseline WER for delta computation
    baseline_wer = l1_df[l1_df["Model"] == MODEL_KEYS["baseline"]][
        ["speaker_native_language", "wer"]].rename(columns={"wer": "wer_baseline"})
    l1_df = l1_df.merge(baseline_wer, on="speaker_native_language", how="left")
    l1_df["wer_rel_imp"] = (l1_df["wer_baseline"] - l1_df["wer"]) / l1_df["wer_baseline"] * 100

    l1s = sorted(l1_df["speaker_native_language"].unique())

    # Grouped bar: WER by L1
    fig = go.Figure()
    for label in MODEL_KEYS.values():
        sub = l1_df[l1_df["Model"] == label].set_index("speaker_native_language")
        fig.add_trace(go.Bar(
            name=label, x=l1s,
            y=[sub.loc[l, "wer"] if l in sub.index else None for l in l1s],
        ))
    fig.update_layout(title=f"{split.capitalize()} — WER by L1", barmode="group",
                      legend=dict(orientation="h", y=1.1, xanchor="center", x=0.5))
    fig.update_yaxes(title_text="WER", tickformat=".0%")
    fig.show()

    # Save per-L1 comparison CSV
    out = Path(RESULTS_DIR) / f"comparison_{split}_by_l1.csv"
    l1_df.to_csv(out, index=False)
    print(f"Saved {out}")
    display(l1_df.sort_values(["speaker_native_language", "Model"])
            .style.format({"wer": "{:.4f}", "cer": "{:.4f}", "wer_rel_imp": "{:+.1f}"})
            .set_caption(f"{split.capitalize()} — Per-L1"))


Saved results/model_perf_comparison/comparison_scripted_by_l1.csv


,speaker_native_language,num_utts,wer,cer,Model,wer_baseline,wer_rel_imp
12,Arabic,150,0.1295,0.0698,CTC Aux,0.145672,+11.1
6,Arabic,150,0.0823,0.0443,Naive LoRA FT,0.145672,+43.5
0,Arabic,150,0.1457,0.0765,Zero-shot,0.145672,+0.0
13,Chinese,150,0.1167,0.0611,CTC Aux,0.121641,+4.1
7,Chinese,150,0.0792,0.0404,Naive LoRA FT,0.121641,+34.9
1,Chinese,150,0.1216,0.0658,Zero-shot,0.121641,+0.0
14,Hindi,150,0.0233,0.0113,CTC Aux,0.035336,+34.0
8,Hindi,150,0.0071,0.0029,Naive LoRA FT,0.035336,+80.0
2,Hindi,150,0.0353,0.0158,Zero-shot,0.035336,+0.0
15,Korean,150,0.0513,0.0253,CTC Aux,0.054131,+5.3


Saved results/model_perf_comparison/comparison_spontaneous_by_l1.csv


,speaker_native_language,num_utts,wer,cer,Model,wer_baseline,wer_rel_imp
12,Arabic,3,0.5361,0.5241,CTC Aux,0.536082,+0.0
6,Arabic,3,0.6701,0.6665,Naive LoRA FT,0.536082,-25.0
0,Arabic,3,0.5361,0.5257,Zero-shot,0.536082,+0.0
13,Chinese,4,0.6236,0.6046,CTC Aux,0.641818,+2.8
7,Chinese,4,0.6145,0.5904,Naive LoRA FT,0.641818,+4.2
1,Chinese,4,0.6418,0.6207,Zero-shot,0.641818,+0.0
14,Hindi,3,0.4324,0.4087,CTC Aux,0.426471,-1.4
8,Hindi,3,0.4265,0.4054,Naive LoRA FT,0.426471,+0.0
2,Hindi,3,0.4265,0.4054,Zero-shot,0.426471,+0.0
15,Korean,4,0.7909,0.7869,CTC Aux,0.802817,+1.5


# Part 3 — Scripted vs Spontaneous Gap

In [8]:
# ── Scripted vs Spontaneous gap per L1 (zero-shot only) ──────────────────────
if "scripted" in results.get("baseline", {}) and "spontaneous" in results.get("baseline", {}):
    s_l1  = compute_grouped_metrics(results["baseline"]["scripted"]).rename(
        columns={"wer": "WER_scripted"})
    sp_l1 = compute_grouped_metrics(results["baseline"]["spontaneous"]).rename(
        columns={"wer": "WER_spontaneous"})
    gap_l1 = s_l1[["speaker_native_language", "WER_scripted"]].merge(
        sp_l1[["speaker_native_language", "WER_spontaneous"]], on="speaker_native_language")
    gap_l1["gap"] = gap_l1["WER_spontaneous"] - gap_l1["WER_scripted"]

    fig = go.Figure()
    l1s_gap = gap_l1["speaker_native_language"].tolist()
    fig.add_trace(go.Bar(name="Scripted",    x=l1s_gap, y=gap_l1["WER_scripted"].tolist()))
    fig.add_trace(go.Bar(name="Spontaneous", x=l1s_gap, y=gap_l1["WER_spontaneous"].tolist()))
    fig.update_layout(title="Zero-shot WER by L1 — Scripted vs Spontaneous", barmode="group",
                      legend=dict(orientation="h", y=1.1, xanchor="center", x=0.5))
    fig.update_yaxes(title_text="WER", tickformat=".0%")
    fig.show()


# Part 4 — Utterance-level Analysis

In [9]:
# ── Utterance WER distributions ──────────────────────────────────────────────
fig = go.Figure()
for key, label in MODEL_KEYS.items():
    for split in SPLITS:
        if split not in results.get(key, {}): continue
        fig.add_trace(go.Histogram(
            x=results[key][split]["utt_wer"],
            name=f"{label} / {split}", opacity=0.5, nbinsx=30,
        ))
fig.update_layout(title="Utterance WER Distribution — All Conditions", barmode="overlay",
                  legend=dict(orientation="h", y=1.1, xanchor="center", x=0.5))
fig.update_xaxes(title_text="Utterance WER")
fig.update_yaxes(title_text="Count")
fig.show()


In [10]:
# ── Worst utterances — zero-shot scripted with cross-model comparison ────────
split = "scripted"
if "baseline" in results and split in results["baseline"]:
    base_df = results["baseline"][split]
    worst   = base_df.nlargest(10, "utt_wer")[
        ["text", "speaker_native_language", "reference_norm", "prediction_norm", "utt_wer"]
    ].rename(columns={"prediction_norm": "pred_baseline", "utt_wer": "wer_baseline"})

    for key, label in MODEL_KEYS.items():
        if key == "baseline" or split not in results.get(key, {}): continue
        other = results[key][split]
        col   = label.replace(" ", "_").lower()
        worst[f"pred_{col}"] = other.loc[worst.index, "prediction_norm"].values
        worst[f"wer_{col}"]  = other.loc[worst.index, "utt_wer"].values

    display(worst.style.format({c: "{:.2f}" for c in worst.columns if c.startswith("wer_")})
            .set_caption(f"Top-10 Worst Utterances (Zero-shot / {split})"))


,text,speaker_native_language,reference_norm,pred_baseline,wer_baseline,pred_naive_lora_ft,wer_naive_lora_ft,pred_ctc_aux,wer_ctc_aux
873,outsiders are allowed five minute speeches the sick man urged,Vietnamese,outsiders are allowed five minute speeches the sick man urged,outsider has loud feminist speech the sixth manner,0.90,outsider as love feminists speech the six men arc,0.90,outside are as loud feminist speech the sixth manorque,0.80
808,the night glow was treacherous to shoot by,Vietnamese,the night glow was treacherous to shoot by,the nightclothes stretcher as you should bite,0.88,the night glow was treacherous you should buy,0.38,the night glow was stretching as you should bite,0.62
802,it was a large canoe,Vietnamese,it was a large canoe,its what a large kind of,0.80,it was a large canoe,0.00,it was a large tunnel,0.20
734,his slim hands gripped the edges of the table,Spanish,his slim hands gripped the edges of the table,this is lim hans repeat date of the day,0.78,his slim hands were the edges of the table,0.11,its the limp hands riped the edge of the table,0.56
786,suddenly his fingers closed tightly over the handkerchief,Vietnamese,suddenly his fingers closed tightly over the handkerchief,shudderly his finger closed tightly over the handcuff of the ship,0.75,suddenly his fingers closed tightly over the hand of his head,0.50,shudderly his finger closed tightly over the handcuffer shaft,0.50
730,the river bared its bosom and snorting steamboats challenged the wilderness,Spanish,the river bared its bosom and snorting steamboats challenged the wilderness,the riverbed is possum and smarthing steam boats challenging the wilderness,0.73,the river bathed in bosom and smelting steam boats challenging the wilderness,0.55,the riverbed is bosom and smarthing steamboats challenging the wilderness,0.45
872,that is what distinguishes all of us from the lower animals,Vietnamese,that is what distinguishes all of us from the lower animals,thus it was distinguished from the lower animal,0.73,thats what distinguishes all the birds from the lower animal,0.45,thus it was distinguished on the first from the lower animal,0.73
95,providence had delivered him through the maelstrom,Arabic,providence had delivered him through the maelstrom,providence had delivered the hams rose the most strong,0.71,providence had delivered him through the maelstrom,0.00,providence had delivered the hams rose the most strong,0.71
611,it fairly clubbed me into recognizing it,Spanish,it fairly clubbed me into recognizing it,its fairly clappin me in to recognize it,0.71,it fairly clubbed me in to recognize it,0.43,it finally clapped me in to recognize it,0.71
846,he obeyed the pressure of her hand,Vietnamese,he obeyed the pressure of her hand,hes all made the pressure of hélène,0.71,he is always the pressure of his hand,0.43,hes obeyed the pressure of his hand,0.29
